In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

# from src.models.update import Update, neighbours
# # from src.models.graph_inference import Graph as Graph_interference
# from src.models.graph_training import Graph as Graph_train
# from src.models.bundle_adjustment import BundleAdjustment
from src.models.dpso import DPSO_train as DPSO

from src.data_loader.data_module_lightning import SonarSimDataModule

In [19]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'


root_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data'
batch_size = 2
num_workers = 0
transforms = None
dm = SonarSimDataModule(root_dir, batch_size, num_workers, transforms, train_config_pth)
dm.setup()
train_data_loader = dm.train_dataloader()

batch = next(iter(train_data_loader))
series, time, trajectory, depth = batch

print(series.shape)
print(time.shape)
print(trajectory.shape)
print(depth.shape)



torch.Size([2, 5, 1, 800, 768])
torch.Size([2, 5, 1])
torch.Size([2, 5, 7])
torch.Size([2, 5, 1])


In [20]:
# # test data 
# start_num = 200
# data_pth_root = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/train/seq_1/fls' # 200.png'

# tmp_img = cv2.imread(os.path.join(data_pth_root, f'0.png'), 0)
# h, w = tmp_img.shape

# frames_in_series = 5
# batch_size = 3
# t_0 = 1.0
# dt = 0.5

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_in_series)], device = device, dtype = torch.float)
# time_tensor = time_tensor.unsqueeze(0)
# time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)

# # read frame 

# frame_sim_b = torch.zeros((batch_size, frames_in_series, 1, h, w))

# for j in range(batch_size):
#     for i in range(frames_in_series):
#         pth = os.path.join(data_pth_root, f'{start_num+batch_size*j+i}.png')
#         frame_sim = cv2.imread(pth, 0)
#         frame_sim = frame_sim.astype(np.uint8)
        
#         frame_sim_b[j, i, :, :] = torch.from_numpy(frame_sim).unsqueeze(0).float()


# print('-'*80)
# print(f'Input data format:')
# print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
# print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
# print(f'time tensor: {time_tensor.shape}')
# print('-'*80)

# **Training test**

In [21]:
dpso_t = DPSO(model_config_pth, sonar_config_pth, train_config_pth)

## forward:

In [22]:
# output = dpso_t(frame_sim_b, time_tensor)

In [23]:
# print(output.shape)


## backward:

In [24]:
optimizer = torch.optim.Adam(dpso_t.parameters(), lr=1e-4)
optimizer.zero_grad()


# series, time, trajectory, depth = batch

pred = dpso_t(series, time, trajectory, depth)


for i in range(len(pred)):

    err, poses = pred[i]
    print(f'err: {err.shape}')
    print(f'poses: {poses.shape}')



# gt = torch.zeros_like(pred.clone())

# criterion = nn.MSELoss()
# loss = criterion(pred, gt)

# print(f'loss: {loss}')

# loss.backward()

# optimizer.step()

debug: poses: torch.Size([2, 5, 7])
debug: poses: torch.Size([10, 7])
torch.Size([180])


RuntimeError: cannot reshape tensor of 0 elements into shape [0, -1] because the unspecified dimension size -1 can be any value and is ambiguous

In [ ]:
# print(pred)

tensor([[[-1.4639e+00,  4.4828e-01, -9.1887e+00,  5.9859e-01, -2.5476e-01,
           7.4396e-01,  1.5269e-01],
         [-4.3462e+00, -3.8534e+00, -1.1149e+00,  6.0354e-01, -6.5354e-01,
          -8.9503e-02, -4.4790e-01],
         [ 1.0261e+00,  2.1350e+00, -5.1903e-02, -2.1338e-01, -3.3623e-01,
          -2.8412e-01, -8.7217e-01],
         [-1.0147e-01,  1.3900e+00,  2.3871e-01,  4.4419e-01, -2.6067e-01,
          -1.2945e-01,  8.4734e-01],
         [-1.4445e+00,  2.9134e+00, -3.9141e+00,  2.9461e-01, -1.6888e-01,
           5.3968e-01,  7.7035e-01]],

        [[-1.0542e+00, -1.5977e+00, -1.1465e-01, -1.0833e-01,  3.9679e-01,
           6.9609e-03,  9.1146e-01],
         [-1.2746e+00, -3.2775e+00, -1.2194e+00,  5.5071e-01, -5.0780e-01,
          -6.2360e-02,  6.5953e-01],
         [ 3.8191e+00, -1.5111e+00, -3.0064e+00,  2.5546e-01, -2.4814e-02,
          -9.5016e-01,  1.7699e-01],
         [-1.2710e+00, -2.4415e+00, -1.9844e+00,  9.1880e-01,  2.3481e-01,
           2.3923e-01,  2.0

# **Inference test**

In [ ]:
# dpso_i = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

# # dpso_i.eval()

DPSO(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), str

## forward:

In [ ]:
# b, n, c, h, w = frame_sim_b.shape
# print(f'Batch size: {b}, frames per batch: {n}')

# prev_t = 0
# # with torch.inference_mode(): # DO NOT USE THIS! this mode doesn't allow to turn grad on and it kills bundle adjustment 
# with torch.no_grad():
#     for batch in range(b):
#         for frame_num in range(n):


#             frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
#             t = time_tensor[batch, frame_num] + prev_t


#             pos, time, nu = dpso_i(frame, t)
#             print(f't: {time}, n: {nu}, pos: {pos}')
#         prev_t = t

#     dpso_i.close()

Batch size: 3, frames per batch: 5
graph append: 0.9490702152252197
calc corr: 0.00652766227722168
calc corr: 0.005514383316040039
calc corr: 0.004566669464111328
calc corr: 0.004522085189819336
calc corr: 0.003770112991333008
t: 1.0, n: 1, pos: tensor([0., 0., 0., 0., 0., 0., 1.])
graph append: 0.9136166572570801
calc corr: 0.009313106536865234
Update operator: 0.005025148391723633
Init BA: 0.0037555694580078125
Run BA: 0.3576483726501465
calc corr: 0.011787891387939453
Update operator: 0.0045850276947021484
Init BA: 0.0030264854431152344
Run BA: 0.3505890369415283
calc corr: 0.005464076995849609
calc corr: 0.005126476287841797
calc corr: 0.00402522087097168
t: 1.5, n: 2, pos: tensor([ 0.8298,  0.1470,  0.1459, -0.0148, -0.0062,  0.0908,  0.9957])
graph append: 0.8977267742156982
calc corr: 0.012584209442138672
Update operator: 0.004519224166870117
Init BA: 0.003511667251586914
Run BA: 0.360384464263916
calc corr: 0.014079570770263672
Update operator: 0.00595545768737793
Init BA: 0.00

In [ ]:
# ckpt_pth = os.path.join('C:/Users/janis/Projekty/Magisterka/SonarOdometry/test', 'test_model.cpkt')
# torch.save(dpso_t.state_dict(), ckpt_pth)

# model_inference = DPSO(model_config_pth, sonar_config_pth, train_config_pth, mode = 'inference', output_dir = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/test')

# model_inference.load_state_dict(torch.load(ckpt_pth, weights_only=False))
# model_inference.eval()

DPSO(
  (PatchGraph): Graph(
    (patchifier): Patchifier(
      (feature_extractor): Encoder(
        (conv1): Conv2d(1, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
        (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (ResBlock1): Sequential(
          (0): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
            (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          )
          (1): ResidualBlock(
            (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (conv2): Conv2d(32, 32, kernel_size=(3, 3), str